# Step-by-Step Walkthrough: BLEU, ROUGE, and METEOR

This notebook provides detailed, hand-computed calculations for three widely used **text generation evaluation metrics**. These metrics are commonly used to evaluate the quality of machine-generated text by comparing it against one or more human-written reference texts.

| Metric | Full Name | Measures | Key Idea |
|--------|-----------|----------|----------|
| **BLEU** | Bilingual Evaluation Understudy | Precision | How much of the *candidate* appears in the *reference*? |
| **ROUGE** | Recall-Oriented Understudy for Gisting Evaluation | Recall (primarily) | How much of the *reference* is captured by the *candidate*? |
| **METEOR** | Metric for Evaluation of Translation with Explicit ORdering | Harmonic mean (recall-weighted) | Like ROUGE but also handles synonyms and penalizes bad word order |

In [ ]:
from collections import Counter
import math

In [ ]:
def get_ngrams(tokens, n):
    """Extract n-grams from token list."""
    ngrams_list = []
    for i in range(len(tokens) - n + 1):
        ngrams_list.append(tuple(tokens[i:i+n]))
    return ngrams_list

---
## Part 1: BLEU Score

### What is BLEU?

**BLEU (Bilingual Evaluation Understudy)** was originally designed for evaluating machine translation but is now used broadly for any text generation task.

**Core idea:** BLEU measures **precision** — what fraction of the n-grams in the *candidate* (generated) text also appear in the *reference* (ground truth) text?

### How BLEU is calculated:

1. **Extract n-grams** (1-gram through 4-gram) from both reference and candidate
2. **Compute modified precision** for each n-gram level: count how many candidate n-grams appear in the reference (clipped to reference counts)
3. **Geometric mean** of the n-gram precisions (equally weighted for BLEU-4)
4. **Brevity penalty (BP)** to penalize candidates shorter than the reference:
   - If candidate length $\geq$ reference length: $BP = 1$
   - Otherwise: $BP = e^{(1 - r/c)}$ where $r$ = reference length, $c$ = candidate length
5. **Final score:** $BLEU = BP \times \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$

### Key properties:
- **Precision-oriented** — focuses on whether the candidate's words/phrases are correct
- **Range:** 0 to 1 (higher is better)
- A score of **1.0** means perfect n-gram overlap
- The brevity penalty ensures you can't cheat by outputting very short but precise text

### Example 1: Partial Match

In [ ]:
reference = "the cat is on the mat"
candidate = "the cat on the mat"

print(f"Reference: '{reference}'")
print(f"Candidate: '{candidate}'")

ref_tokens = reference.split()
cand_tokens = candidate.split()

print(f"\nStep 0: Tokenize")
print(f"  Reference tokens: {ref_tokens}")
print(f"  Candidate tokens: {cand_tokens}")
print(f"  Ref length: {len(ref_tokens)}, Cand length: {len(cand_tokens)}")

#### Step 1: Extract and Compare N-grams

An **n-gram** is a contiguous sequence of *n* tokens. For BLEU-4 we compute precision for 1-grams through 4-grams.

**Modified precision** clips each candidate n-gram count to its maximum count in the reference, preventing inflated scores from repeated words.

In [ ]:
for n in range(1, 5):
    ref_ngrams = get_ngrams(ref_tokens, n)
    cand_ngrams = get_ngrams(cand_tokens, n)

    print(f"{n}-grams:")
    print(f"  Reference {n}-grams: {ref_ngrams}")
    print(f"  Candidate {n}-grams: {cand_ngrams}")

    # Count matches (clipped to reference counts)
    ref_counter = Counter(ref_ngrams)
    cand_counter = Counter(cand_ngrams)

    matching_count = sum((cand_counter & ref_counter).values())
    total_cand = sum(cand_counter.values())

    precision = matching_count / total_cand if total_cand > 0 else 0.0

    print(f"  Matching {n}-grams: {matching_count}")
    print(f"  Total candidate {n}-grams: {total_cand}")
    print(f"  {n}-gram precision: {matching_count}/{total_cand} = {precision:.4f}")
    print()

#### Step 2: Geometric Mean of Precisions

BLEU-4 uses **equal weights** $(w_1 = w_2 = w_3 = w_4 = 0.25)$ and computes the geometric mean:

$$\text{Geometric Mean} = (P_1 \times P_2 \times P_3 \times P_4)^{1/4}$$

The geometric mean is used (rather than arithmetic mean) because it heavily penalizes any single n-gram precision being zero — if *any* precision is 0, the entire score is 0.

In [ ]:
precisions = []
for n in range(1, 5):
    ref_ngrams = Counter(get_ngrams(ref_tokens, n))
    cand_ngrams = Counter(get_ngrams(cand_tokens, n))
    matching = sum((cand_ngrams & ref_ngrams).values())
    total = sum(cand_ngrams.values())
    prec = matching / total if total > 0 else 0.0
    precisions.append(prec)
    print(f"  P{n} = {prec:.4f}")

print(f"\n  Geometric mean = (P1 * P2 * P3 * P4)^(1/4)")
print(f"                 = ({precisions[0]:.4f} * {precisions[1]:.4f} * {precisions[2]:.4f} * {precisions[3]:.4f})^(1/4)")

if all(p > 0 for p in precisions):
    log_sum = sum(math.log(p) for p in precisions)
    geometric_mean = math.exp(log_sum / 4)
    print(f"                 = exp(({log_sum:.4f}) / 4)")
    print(f"                 = exp({log_sum/4:.4f})")
    print(f"                 = {geometric_mean:.4f}")
else:
    geometric_mean = 0.0
    print(f"  -> One or more precisions is 0, so geometric mean = 0")

#### Step 3: Brevity Penalty

The **brevity penalty** prevents gaming the score by generating very short outputs that happen to be precise.

$$BP = \begin{cases} 1 & \text{if } c > r \\ e^{(1 - r/c)} & \text{if } c \leq r \end{cases}$$

where $c$ = candidate length, $r$ = reference length.

In [ ]:
print(f"  Reference length: {len(ref_tokens)}")
print(f"  Candidate length: {len(cand_tokens)}")
print(f"  Ratio: {len(cand_tokens)} / {len(ref_tokens)} = {len(cand_tokens)/len(ref_tokens):.4f}")

ratio = len(cand_tokens) / len(ref_tokens) if len(ref_tokens) > 0 else 0
if ratio < 1:
    brevity_penalty = math.exp(1 - 1/ratio) if ratio > 0 else 0
    print(f"  Ratio < 1, so apply penalty")
    print(f"  BP = exp(1 - 1/{ratio:.4f}) = exp(1 - {1/ratio:.4f}) = exp({1 - 1/ratio:.4f})")
    print(f"     = {brevity_penalty:.4f}")
else:
    brevity_penalty = 1.0
    print(f"  Ratio >= 1, so no penalty (BP = 1.0)")

#### Step 4: Final BLEU Score

$$BLEU = BP \times \text{Geometric Mean}$$

In [ ]:
bleu_score = geometric_mean * brevity_penalty
print(f"BLEU = Geometric Mean x Brevity Penalty")
print(f"     = {geometric_mean:.4f} x {brevity_penalty:.4f}")
print(f"     = {bleu_score:.4f}")

### Example 2: Perfect Match

When the candidate is identical to the reference, all n-gram precisions are 1.0, the geometric mean is 1.0, and no brevity penalty is applied.

In [ ]:
ref2 = "the cat sat on the mat"
cand2 = "the cat sat on the mat"

print(f"Reference: '{ref2}'")
print(f"Candidate: '{cand2}'")

print(f"\nSince the sentences are identical:")
print(f"  All n-grams will match perfectly")
print(f"  All precisions = 1.0")
print(f"  Geometric mean = 1.0")
print(f"  Brevity penalty = 1.0 (same length)")
print(f"  BLEU = 1.0 x 1.0 = 1.0")

### Example 3: No N-gram Overlap

When the candidate shares no tokens with the reference, all precisions are 0. Since the geometric mean of any set containing 0 is 0, the BLEU score is **0.0** regardless of the brevity penalty.

In [ ]:
ref3 = "the dog ran fast"
cand3 = "a cat walked slowly"

print(f"Reference: '{ref3}'")
print(f"Candidate: '{cand3}'")

ref3_tokens = ref3.split()
cand3_tokens = cand3.split()

print(f"\nReference tokens: {ref3_tokens}")
print(f"Candidate tokens: {cand3_tokens}")
print(f"\nNo tokens in common between reference and candidate")
print(f"  1-gram precision: 0/4 = 0.0")
print(f"  2-gram precision: 0/3 = 0.0")
print(f"  etc.")
print(f"\nGeometric mean = (0.0 x 0.0 x 0.0 x 0.0)^(1/4) = 0.0")
print(f"Brevity penalty doesn't matter when precision = 0")
print(f"BLEU = 0.0 x (anything) = 0.0")

---
## Part 2: ROUGE Score

### What is ROUGE?

**ROUGE (Recall-Oriented Understudy for Gisting Evaluation)** was designed for evaluating automatic summarization but is now used for many text generation tasks.

**Core idea:** ROUGE measures **recall** — what fraction of the n-grams in the *reference* text are captured by the *candidate* text? (It also reports precision and F1.)

### ROUGE Variants:

| Variant | What it measures | Sensitive to word order? |
|---------|-----------------|-------------------------|
| **ROUGE-1** | Unigram (single word) overlap | No |
| **ROUGE-2** | Bigram (2-word sequence) overlap | Partially (within bigrams) |
| **ROUGE-L** | Longest Common Subsequence (LCS) | Yes |

### Key formulas:

$$\text{Recall} = \frac{\text{matching n-grams}}{\text{total reference n-grams}}$$

$$\text{Precision} = \frac{\text{matching n-grams}}{\text{total candidate n-grams}}$$

$$F_1 = \frac{2 \times Precision \times Recall}{Precision + Recall}$$

### BLEU vs ROUGE:
- **BLEU** asks: "Is everything the candidate said correct?" (precision)
- **ROUGE** asks: "Did the candidate cover everything in the reference?" (recall)

### ROUGE-1: Unigram Overlap

In [ ]:
reference = "the cat is on the mat"
candidate = "the cat on the mat"

print(f"Reference: '{reference}'")
print(f"Candidate: '{candidate}'")

ref_tokens = reference.split()
cand_tokens = candidate.split()

print(f"\nStep 1: Create word counters")
ref_counter = Counter(ref_tokens)
cand_counter = Counter(cand_tokens)
print(f"  Reference word counts: {dict(ref_counter)}")
print(f"  Candidate word counts: {dict(cand_counter)}")

print(f"\nStep 2: Find common words (intersection with min counts)")
common = cand_counter & ref_counter
print(f"  Common words: {dict(common)}")
print(f"  Total common: {sum(common.values())}")

print(f"\nStep 3: Calculate ROUGE-1 Recall")
print(f"  Recall = (matching words) / (total reference words)")
matching_words = sum(common.values())
ref_total = sum(ref_counter.values())
recall = matching_words / ref_total if ref_total > 0 else 0.0
print(f"         = {matching_words} / {ref_total}")
print(f"         = {recall:.4f}")

print(f"\nStep 4: Calculate ROUGE-1 Precision")
print(f"  Precision = (matching words) / (total candidate words)")
cand_total = sum(cand_counter.values())
precision = matching_words / cand_total if cand_total > 0 else 0.0
print(f"           = {matching_words} / {cand_total}")
print(f"           = {precision:.4f}")

print(f"\nStep 5: Calculate ROUGE-1 F1 Score")
if precision + recall > 0:
    f1 = 2 * (precision * recall) / (precision + recall)
else:
    f1 = 0.0
print(f"  F1 = 2 x (Precision x Recall) / (Precision + Recall)")
print(f"     = 2 x ({precision:.4f} x {recall:.4f}) / ({precision:.4f} + {recall:.4f})")
print(f"     = 2 x {precision * recall:.4f} / {precision + recall:.4f}")
print(f"     = {f1:.4f}")

### ROUGE-2: Bigram Overlap

ROUGE-2 works the same way as ROUGE-1, but uses **bigrams** (2-word sequences) instead of unigrams. This partially captures word order since adjacent word pairs must match.

In [ ]:
print(f"Step 1: Extract bigrams")
ref_bigrams = get_ngrams(ref_tokens, 2)
cand_bigrams = get_ngrams(cand_tokens, 2)
print(f"  Reference bigrams: {ref_bigrams}")
print(f"  Candidate bigrams: {cand_bigrams}")

print(f"\nStep 2: Create bigram counters")
ref_bigram_counter = Counter(ref_bigrams)
cand_bigram_counter = Counter(cand_bigrams)
print(f"  Reference bigram counts: {dict(ref_bigram_counter)}")
print(f"  Candidate bigram counts: {dict(cand_bigram_counter)}")

print(f"\nStep 3: Find matching bigrams")
common_bigrams = cand_bigram_counter & ref_bigram_counter
print(f"  Common bigrams: {dict(common_bigrams)}")
matching_bigrams = sum(common_bigrams.values())
print(f"  Total matching: {matching_bigrams}")

print(f"\nStep 4: Calculate ROUGE-2")
ref_bigram_total = sum(ref_bigram_counter.values())
cand_bigram_total = sum(cand_bigram_counter.values())
recall_2 = matching_bigrams / ref_bigram_total if ref_bigram_total > 0 else 0.0
precision_2 = matching_bigrams / cand_bigram_total if cand_bigram_total > 0 else 0.0
f1_2 = 2 * (precision_2 * recall_2) / (precision_2 + recall_2) if (precision_2 + recall_2) > 0 else 0.0
print(f"  Recall:    {matching_bigrams} / {ref_bigram_total} = {recall_2:.4f}")
print(f"  Precision: {matching_bigrams} / {cand_bigram_total} = {precision_2:.4f}")
print(f"  F1:        {f1_2:.4f}")

### ROUGE-L: Longest Common Subsequence (LCS)

**ROUGE-L** uses the **Longest Common Subsequence** — the longest sequence of tokens that appears in both texts *in the same order*, but not necessarily contiguously.

**Key difference from n-gram methods:** LCS doesn't require matches to be adjacent. It captures sentence-level word order similarity.

**Example:**
- Reference: `the cat is on the mat`
- Candidate: `the cat on the mat`
- LCS: `the cat on the mat` (length 5) — "is" is skipped

LCS is computed using **dynamic programming** with the recurrence:

$$LCS(i,j) = \begin{cases} LCS(i-1,j-1) + 1 & \text{if } ref_i = cand_j \\ \max(LCS(i-1,j),\ LCS(i,j-1)) & \text{otherwise} \end{cases}$$

In [ ]:
print(f"Reference: {ref_tokens}")
print(f"Candidate: {cand_tokens}")

print(f"\nStep 1: Build LCS Matrix using Dynamic Programming")
m, n = len(ref_tokens), len(cand_tokens)
lcs_matrix = [[0] * (n + 1) for _ in range(m + 1)]

print(f"  Rows (reference): {m}, Columns (candidate): {n}")
print(f"  For each cell [i,j]:")
print(f"    If ref[i-1] == cand[j-1]: matrix[i][j] = matrix[i-1][j-1] + 1")
print(f"    Else: matrix[i][j] = max(matrix[i-1][j], matrix[i][j-1])")

for i in range(1, m + 1):
    for j in range(1, n + 1):
        if ref_tokens[i-1] == cand_tokens[j-1]:
            lcs_matrix[i][j] = lcs_matrix[i-1][j-1] + 1
        else:
            lcs_matrix[i][j] = max(lcs_matrix[i-1][j], lcs_matrix[i][j-1])

print(f"\n  Final LCS Matrix:")
print(f"       ", " ".join(f"{t:>3}" for t in [""] + cand_tokens))
for i, ref_token in enumerate([""] + ref_tokens):
    row = " ".join(f"{val:>3}" for val in lcs_matrix[i])
    print(f"  {ref_token:>3}  {row}")

lcs_length = lcs_matrix[m][n]
print(f"\n  LCS length = {lcs_length}")

In [ ]:
print(f"Calculate ROUGE-L")
recall_l = lcs_length / len(ref_tokens) if len(ref_tokens) > 0 else 0.0
precision_l = lcs_length / len(cand_tokens) if len(cand_tokens) > 0 else 0.0
f1_l = 2 * (precision_l * recall_l) / (precision_l + recall_l) if (precision_l + recall_l) > 0 else 0.0
print(f"  Recall:    {lcs_length} / {len(ref_tokens)} = {recall_l:.4f}")
print(f"  Precision: {lcs_length} / {len(cand_tokens)} = {precision_l:.4f}")
print(f"  F1:        {f1_l:.4f}")

print(f"\nROUGE-L is order-aware:")
print(f"  LCS preserves word order in the sequence")
print(f"  If words appear in different order, LCS is shorter")

### ROUGE Example 2: Different Word Order

This example demonstrates the key difference between ROUGE-1 and ROUGE-L. When the same words appear in a different order:
- **ROUGE-1** remains high (same words present)
- **ROUGE-L** drops (word order differs, so LCS is shorter)

In [ ]:
ref_ex2 = "alice gave a book to bob"
cand_ex2 = "bob received a book from alice"

print(f"Reference: '{ref_ex2}'")
print(f"Candidate: '{cand_ex2}'")

ref_ex2_tokens = ref_ex2.split()
cand_ex2_tokens = cand_ex2.split()

print(f"\nReference tokens: {ref_ex2_tokens}")
print(f"Candidate tokens: {cand_ex2_tokens}")

# ROUGE-1
ref_ex2_counter = Counter(ref_ex2_tokens)
cand_ex2_counter = Counter(cand_ex2_tokens)
common_ex2 = cand_ex2_counter & ref_ex2_counter
matching_ex2 = sum(common_ex2.values())
print(f"\nCommon words: {dict(common_ex2)}")
print(f"Total matching: {matching_ex2}")

recall_ex2_1 = matching_ex2 / len(ref_ex2_tokens)
precision_ex2_1 = matching_ex2 / len(cand_ex2_tokens)
f1_ex2_1 = 2 * (precision_ex2_1 * recall_ex2_1) / (precision_ex2_1 + recall_ex2_1)

print(f"ROUGE-1 F1: {f1_ex2_1:.4f}")

# ROUGE-L
m2, n2 = len(ref_ex2_tokens), len(cand_ex2_tokens)
lcs_matrix_2 = [[0] * (n2 + 1) for _ in range(m2 + 1)]
for i in range(1, m2 + 1):
    for j in range(1, n2 + 1):
        if ref_ex2_tokens[i-1] == cand_ex2_tokens[j-1]:
            lcs_matrix_2[i][j] = lcs_matrix_2[i-1][j-1] + 1
        else:
            lcs_matrix_2[i][j] = max(lcs_matrix_2[i-1][j], lcs_matrix_2[i][j-1])

lcs_length_2 = lcs_matrix_2[m2][n2]
recall_ex2_l = lcs_length_2 / len(ref_ex2_tokens)
precision_ex2_l = lcs_length_2 / len(cand_ex2_tokens)
f1_ex2_l = 2 * (precision_ex2_l * recall_ex2_l) / (precision_ex2_l + recall_ex2_l)

print(f"ROUGE-L F1: {f1_ex2_l:.4f}")
print(f"\n-> Notice: ROUGE-1 is high (same words) but ROUGE-L is lower (different order)")
print(f"   ROUGE-1 doesn't care about word order, ROUGE-L does!")

---
## Part 3: METEOR Score

### What is METEOR?

**METEOR (Metric for Evaluation of Translation with Explicit ORdering)** was designed to address several shortcomings of BLEU.

### How METEOR differs from BLEU and ROUGE:

| Feature | BLEU | ROUGE | METEOR |
|---------|------|-------|--------|
| Primary focus | Precision | Recall | Recall-weighted harmonic mean |
| Synonym matching | No | No | **Yes** |
| Stemming | No | No | **Yes** (e.g., "running" matches "runs") |
| Word order penalty | No (implicit via n-grams) | Only in ROUGE-L | **Explicit penalty** |
| Correlates with human judgment | Good | Good | **Better** |

### How METEOR is calculated:

1. **Find matches** between candidate and reference tokens (exact, stem, synonym)
2. **Compute precision and recall** from the matches
3. **Harmonic mean** with $\beta = 3$ (recall weighted 3x more than precision):
   $$F = \frac{(1 + \beta^2) \cdot P \cdot R}{\beta^2 \cdot P + R} = \frac{10 \cdot P \cdot R}{9P + R}$$
4. **Word order penalty:** fragmentation penalty reduces the score if matched words are not in contiguous chunks:
   $$\text{Penalty} = 0.5 \times \left(\frac{\text{\# chunks}}{\text{\# matches}}\right)$$
   $$\text{METEOR} = F \times (1 - \text{Penalty})$$

### Example 1: Basic METEOR Calculation

In [ ]:
reference = "the cat is on the mat"
candidate = "the cat on the mat"

print(f"Reference: '{reference}'")
print(f"Candidate: '{candidate}'")

ref_tokens = reference.split()
cand_tokens = candidate.split()

print(f"\nReference tokens: {ref_tokens}")
print(f"Candidate tokens: {cand_tokens}")

print(f"\nStep 1: Find Exact Token Matches")
ref_set = set(ref_tokens)
cand_set = set(cand_tokens)
matches = ref_set & cand_set
print(f"  Reference tokens (unique): {ref_set}")
print(f"  Candidate tokens (unique): {cand_set}")
print(f"  Matches: {matches}")
print(f"  Number of matches: {len(matches)}")

print(f"\nStep 2: Calculate Precision and Recall")
precision = len(matches) / len(cand_tokens) if len(cand_tokens) > 0 else 0.0
recall = len(matches) / len(ref_tokens) if len(ref_tokens) > 0 else 0.0
print(f"  Precision = matches / candidate tokens = {len(matches)} / {len(cand_tokens)} = {precision:.4f}")
print(f"  Recall    = matches / reference tokens = {len(matches)} / {len(ref_tokens)} = {recall:.4f}")

#### Step 3: Harmonic Mean with Recall Emphasis ($\beta = 3$)

Unlike the standard F1 score (which weights precision and recall equally), METEOR uses $\beta = 3$ so that **recall is 3 times more important** than precision. This reflects the intuition that capturing the reference content matters more than avoiding extra words.

$$F = \frac{(1 + \beta^2) \cdot P \cdot R}{\beta^2 \cdot P + R} = \frac{10 \cdot P \cdot R}{9P + R}$$

In [ ]:
alpha = 3.0
if precision + recall > 0:
    f_score = (1 + alpha**2) * (precision * recall) / ((alpha**2 * precision) + recall)
else:
    f_score = 0.0
print(f"F-score = ((1 + 9) x {precision:.4f} x {recall:.4f}) / (9 x {precision:.4f} + {recall:.4f})")
print(f"        = (10 x {precision * recall:.4f}) / ({9*precision + recall:.4f})")
print(f"        = {10 * precision * recall:.4f} / {9*precision + recall:.4f}")
print(f"        = {f_score:.4f}")

#### Step 4: Word Order Penalty

METEOR penalizes fragmented matches. If matching words are scattered (not in contiguous chunks), the score is reduced.

$$\text{Penalty} = 0.5 \times \frac{\text{\# chunks}}{\text{\# matches}}$$

- A **chunk** is a maximal contiguous sequence of matched words (in both reference and candidate order)
- Fewer chunks = better word order = lower penalty
- If all matches form one chunk, penalty = $0.5 \times 1/n$ (minimal)
- If every match is its own chunk, penalty = $0.5$ (maximum)

In [ ]:
print(f"For this example, most matches are contiguous, so penalty is minimal.")

meteor_score = f_score  # Simplified - ignoring detailed penalty for clarity
print(f"\nFinal METEOR Score (simplified, without detailed penalty): {f_score:.4f}")

### Example 2: Why Synonyms Matter

This is METEOR's biggest advantage over BLEU and ROUGE. Consider two sentences that mean the same thing but use different words:
- "the **quick** brown fox **jumps**"
- "the **fast** brown fox **leaps**"

BLEU and ROUGE would penalize this because "quick" ≠ "fast" and "jumps" ≠ "leaps" at the token level. METEOR recognizes these as **synonyms** and gives full credit.

In [ ]:
ref_syn = "the quick brown fox jumps"
cand_syn = "the fast brown fox leaps"

print(f"Reference: '{ref_syn}'")
print(f"Candidate: '{cand_syn}'")

ref_syn_tokens = ref_syn.split()
cand_syn_tokens = cand_syn.split()

print(f"\nReference tokens: {ref_syn_tokens}")
print(f"Candidate tokens: {cand_syn_tokens}")

# Exact match only
matches_exact = set(ref_syn_tokens) & set(cand_syn_tokens)
print(f"\nExact matches: {matches_exact}")
print(f"Count: {len(matches_exact)}")

prec_exact = len(matches_exact) / len(cand_syn_tokens)
recall_exact = len(matches_exact) / len(ref_syn_tokens)
f_exact = (1 + 9) * (prec_exact * recall_exact) / (9*prec_exact + recall_exact) if (prec_exact + recall_exact) > 0 else 0

print(f"\nExact-match METEOR:")
print(f"  Precision: {len(matches_exact)}/{len(cand_syn_tokens)} = {prec_exact:.4f}")
print(f"  Recall:    {len(matches_exact)}/{len(ref_syn_tokens)} = {recall_exact:.4f}")
print(f"  METEOR:    {f_exact:.4f}")

print(f"\nFull METEOR (with synonym matching):")
print(f"  'quick' matches 'fast' (synonym)")
print(f"  'jumps' matches 'leaps' (synonym)")
print(f"  So actual matches would be: the, brown, fox, quick<->fast, jumps<->leaps")
print(f"  With synonyms: 5/5 = 1.0 (perfect score)")
print(f"\n-> This is why METEOR is better for paraphrasing:")
print(f"   It recognizes that 'quick' and 'fast' mean the same thing")

### Example 3: Word Order Penalty in Action

Even when all tokens match exactly, METEOR penalizes **scrambled word order**. This makes METEOR more sensitive to fluency and grammaticality than simple token-overlap metrics.

In [ ]:
ref_order = "alice gave book to bob"
cand_order = "bob book gave alice to"

print(f"Reference: '{ref_order}'")
print(f"Candidate: '{cand_order}'")

ref_order_tokens = ref_order.split()
cand_order_tokens = cand_order.split()

print(f"\nAll tokens match, but order is scrambled!")

matches_order = set(ref_order_tokens) & set(cand_order_tokens)
print(f"Matching tokens: {matches_order} (count: {len(matches_order)})")

# Precision and recall
prec_order = len(matches_order) / len(cand_order_tokens)
recall_order = len(matches_order) / len(ref_order_tokens)
f_order = (1 + 9) * (prec_order * recall_order) / (9*prec_order + recall_order) if (prec_order + recall_order) > 0 else 0

print(f"\nBefore word order penalty:")
print(f"  Precision: {len(matches_order)}/{len(cand_order_tokens)} = {prec_order:.4f}")
print(f"  Recall:    {len(matches_order)}/{len(ref_order_tokens)} = {recall_order:.4f}")
print(f"  F-score:   {f_order:.4f}")

print(f"\nWord order penalty:")
print(f"  Reference word order: alice -> gave -> book -> to -> bob")
print(f"  Candidate word order: bob -> book -> gave -> alice -> to")
print(f"  These are in completely different order")
print(f"  Penalty would be large: penalty = 0.5 x (# chunks) / (# matches)")
print(f"  # chunks = # of contiguous matching sequences = high")
print(f"  Final METEOR = F-score x (1 - penalty) = {f_order:.4f} x (smaller multiplier)")
print(f"  Resulting METEOR << {f_order:.4f}")

print(f"\n-> Key insight: METEOR punishes scrambled word order!")
print(f"   This makes it more sophisticated than simple token matching")

---
## Summary: When to Use Each Metric

| Metric | Best For | Strengths | Weaknesses |
|--------|----------|-----------|------------|
| **BLEU** | Machine translation, short text generation | Fast, well-understood, penalizes short outputs | No synonym handling, precision-only focus |
| **ROUGE** | Summarization, longer text generation | Recall-focused (did we capture the key content?), multiple variants | No synonym handling, ROUGE-1 ignores word order |
| **METEOR** | Translation, paraphrase detection | Synonym/stem matching, word order penalty, best correlation with humans | Slower (needs WordNet), more complex |

### General guidance:
- Use **BLEU** when you care about precision ("Is the output correct?")
- Use **ROUGE** when you care about recall ("Did the output cover the key content?")
- Use **METEOR** when you need the most nuanced evaluation, especially with paraphrasing
- In practice, **report multiple metrics** to get a well-rounded view of quality